# President Tweets on Risk Metrics

James 

# Imports

In [22]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import warnings
warnings.filterwarnings('ignore')

# Set plotting style if needed
import matplotlib.pyplot as plt
plt.style.use('ggplot')

## Load data

### Financial Data

In [32]:
# Load financial datasets
vix = pd.read_csv('data/VIX_DAILY.csv', na_values='.')
treasury = pd.read_csv('data/3mo_treasury.csv', na_values='.')
oil = pd.read_csv('data/oil_px.csv')

# Standardize date columns to datetime
vix['date'] = pd.to_datetime(vix['observation_date'])
treasury['date'] = pd.to_datetime(treasury['observation_date'])
oil['date'] = pd.to_datetime(oil['observation_date'])

# Clean and convert values to numeric
vix['VIXCLS'] = pd.to_numeric(vix['VIXCLS'], errors='coerce')
treasury['DGS3MO'] = pd.to_numeric(treasury['DGS3MO'], errors='coerce')
oil['DCOILWTICO'] = pd.to_numeric(oil['DCOILWTICO'], errors='coerce')

# Merge financial data on date
risk_df = vix[['date', 'VIXCLS']].merge(treasury[['date', 'DGS3MO']], on='date', how='outer')
risk_df = risk_df.merge(oil[['date', 'DCOILWTICO']].rename(columns={'DCOILWTICO': 'Oil_Price'}), on='date', how='outer')

# Sort and calculate daily changes (Risk metric is often the daily change or % change)
risk_df = risk_df.sort_values('date').set_index('date')
risk_df['VIX_Change'] = risk_df['VIXCLS'].diff() # Daily absolute change in VIX
risk_df['Oil_Ret'] = risk_df['Oil_Price'].pct_change() # Daily return in Oil
risk_df = risk_df.dropna()

### Twitter Data

In [33]:
# Load Twitter datasets
biden = pd.read_csv('data/JoeBiden.csv')
obama = pd.read_csv('data/obama.csv')
trump = pd.read_csv('data/trump_tweets.csv')

# Standardize text and date columns
biden = biden[['date', 'content']].rename(columns={'content': 'text'})
obama = obama[['Timestamp', 'Text']].rename(columns={'Timestamp': 'date', 'Text': 'text'})
trump = trump[['date', 'text']]

# Add author identifiers
biden['author'] = 'Biden'
obama['author'] = 'Obama'
trump['author'] = 'Trump'

# Combine all tweets
tweets = pd.concat([biden, obama, trump], ignore_index=True)
tweets['date'] = pd.to_datetime(tweets['date'], errors='coerce').dt.date
tweets['date'] = pd.to_datetime(tweets['date'])
tweets = tweets.dropna(subset=['date', 'text'])

# Define presidency dates to create the 'is_president' flag
def check_if_president(row):
    d = row['date']
    author = row['author']
    if author == 'Obama' and ('2009-01-20' <= str(d.date()) < '2017-01-20'):
        return 1
    elif author == 'Trump' and ('2017-01-20' <= str(d.date()) < '2021-01-20'):
        return 1
    elif author == 'Biden' and ('2021-01-20' <= str(d.date())):
        return 1
    return 0

tweets['is_president'] = tweets.apply(check_if_president, axis=1)

# Group tweets by day (concatenate text, take max of is_president)
daily_tweets = tweets.groupby('date').agg({
    'text': lambda x: ' '.join(x.astype(str)),
    'is_president': 'max'
}).reset_index()

### Merge dfs

In [34]:
# Merge the daily tweets with the daily risk metrics
df_merged = risk_df.merge(daily_tweets.set_index('date'), left_index=True, right_index=True, how='inner')

# Clean the combined text: lowercasing, removing URLs, special characters
def clean_text(text):
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation/numbers
    return text.lower()

df_merged['clean_text'] = df_merged['text'].apply(clean_text)

## NLP Text extraction

In [35]:
# Using TF-IDF to extract top 100 most frequent/important keywords to prevent overfitting
vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df_merged['clean_text'])

# Create a DataFrame for the TF-IDF features
keyword_features = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=vectorizer.get_feature_names_out(),
    index=df_merged.index
)

# Join the keyword features back to our main dataset
df_final = pd.concat([df_merged, keyword_features], axis=1)
df_final = df_final.dropna()

### regression

In [36]:
# Define target (Y) and predictors (X)
# Target: Daily Change in the VIX (Volatility Index)
Y = df_final['VIX_Change'] 

# Predictors: Keyword frequencies + the "is_president" effect + a constant intercept
X = df_final[['is_president'] + list(keyword_features.columns)]
X = sm.add_constant(X)

# Fit Ordinary Least Squares (OLS) Regression
model = sm.OLS(Y, X).fit()

# Print the regression summary
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             VIX_Change   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.003
Date:                Wed, 15 Apr 2026   Prob (F-statistic):              0.476
Time:                        13:11:58   Log-Likelihood:                -2470.9
No. Observations:                1136   AIC:                             5146.
Df Residuals:                    1034   BIC:                             5659.
Df Model:                         101                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0184      0.226      0.

## Final words 

In [37]:
# Extract P-values and Coefficients from the model
results_df = pd.DataFrame({
    'Coefficient': model.params,
    'P-Value': model.pvalues
})

# Filter out the constant to focus purely on features
results_df = results_df.drop('const', errors='ignore')

# Determine significance (p < 0.05) and sort by absolute coefficient impact
significant_factors = results_df[results_df['P-Value'] < 0.05]
significant_factors['Abs_Impact'] = significant_factors['Coefficient'].abs()
significant_factors = significant_factors.sort_values(by='Abs_Impact', ascending=False)

print("Top Impactful Words / Features (Statistically Significant):")
display(significant_factors.head(20))

# Specifically check the effect of being in office
print("\nEffect of being President on Risk:")
if 'is_president' in significant_factors.index:
    effect = significant_factors.loc['is_president', 'Coefficient']
    print(f"Statistically significant effect. Coefficient: {effect:.4f}")
else:
    print(f"Not statistically significant at p < 0.05. Coefficient was {results_df.loc['is_president', 'Coefficient']:.4f} (p-value: {results_df.loc['is_president', 'P-Value']:.4f})")

Top Impactful Words / Features (Statistically Significant):


,Coefficient,P-Value,Abs_Impact
demdebate,3.305775,0.000148,3.305775
crisis,-2.414209,0.012838,2.414209
going,-1.994627,0.011053,1.994627
americans,-1.698628,0.013644,1.698628



Effect of being President on Risk:
Not statistically significant at p < 0.05. Coefficient was -0.1605 (p-value: 0.4912)
